# US Equity Interview Cheat Sheet

This notebook is a speaking document for quant researcher interviews. It is designed to help you explain:

- what the repo does
- why each strategy should work
- the math behind the main components
- how the full research pipeline fits together
- where the weaknesses and implementation risks are

## 30-Second Pitch

I built a modular long-short U.S. equity research stack that separates alpha generation, portfolio construction, and risk control. The repo is cross-sectional and implementation-aware: I do not only search for predictive signals, I explicitly control for turnover, concentration, beta exposure, and market regime. That makes the process closer to real portfolio research than a raw signal backtest.

## Repo Map

- `alpha_ma_crossover.py`: moving-average alpha with tail gating and inverse-vol scaling
- `alpha_hmm.py`: 2-state Gaussian HMM regime overlay
- `alpha_fund_momentum.py`: blended price + fundamentals alpha
- `risk_overlays.py`: market-vol, beta-neutral, and drawdown scaling
- `kalman_overlays.py`: Kalman-filtered regime scaling and innovation gating
- `grid_search_pipeline.py`: modular search and evaluation loop
- `quality_kinked.py`: asymmetric quality function for strategy selection


## Full Research Pipeline

1. Build a clean panel of price, volume, and optionally fundamentals.
2. Convert raw observations into alpha scores.
3. Apply cross-sectional gating so the book trades only strong names.
4. Scale by volatility so risk is not dominated by a few unstable names.
5. Convert alpha to portfolio weights and enforce neutrality constraints.
6. Apply regime overlays or drawdown-aware exposure controls.
7. Backtest net returns and track Sharpe, drawdown, turnover, and max weight.
8. Select parameters with a score that rewards return quality but penalizes implementation drag.

## Research Philosophy

The central research idea is that a useful alpha is not just predictive. It must survive portfolio formation.

That means I care about four things simultaneously:

- **signal strength**: does the alpha separate winners from losers?
- **robustness**: is it stable under outliers and regime changes?
- **implementability**: can I trade it without extreme turnover or crowding?
- **risk shape**: is the PnL driven by true stock selection or hidden market exposure?

## Strategy 1: Moving-Average Crossover Alpha

### Signal Definition

$$
S_{i,t}=\frac{\mathrm{SMA}^{short}_{i,t}-\mathrm{SMA}^{long}_{i,t}}{|\mathrm{SMA}^{long}_{i,t}|}
$$

where $i$ indexes stocks and $t$ indexes time.

In code, the signal is built from rolling short and long moving averages, then transformed into a cross-sectional score.

### Intuition

The intuition is classic time-series trend carried into a cross-sectional book. If a stock's short moving average is above its long moving average, it may indicate persistent buying pressure or delayed information incorporation.

Why it can work:

- institutional flows adjust slowly
- earnings revisions and sentiment diffuse gradually
- medium-horizon trends often persist before mean reversion dominates

Why it can fail:

- reversals after sharp one-day moves
- noisy names with unstable prices
- hidden market beta masquerading as stock selection

### Why Tail Gating?

The repo trades only the strongest names each day instead of using the whole cross-section.

Mathematically:

$$
\mathcal{K}_t=\{i:\ |S_{i,t}| \ge q_t(\tau)\}
$$

where $q_t(\tau)$ is a cross-sectional quantile threshold.

Intuition:

- weak signals are mostly noise
- the center of the cross-section creates turnover without much edge
- strong tails usually carry a better signal-to-noise ratio

This is one of the strongest things to say in interview: **I do not assume every score should be traded just because it exists.**

### Why Volatility Scaling?

The repo uses an ATR-like risk scale from rolling absolute returns:

$$
\alpha_{i,t}=\operatorname{sign}(S_{i,t})\cdot \frac{1}{\mathrm{ATR}_{i,t}}
$$

Why this makes sense:

- raw signals otherwise overweight high-volatility names
- inverse-vol scaling equalizes risk contribution across stocks
- without scaling, one unstable stock can dominate the book

Interview line:

> I want a portfolio driven by ideas, not by whichever names happen to be the noisiest.

## Strategy 2: Fundamental Momentum Alpha

### Math

Price momentum leg:

$$
Z^{price}_{i,t}=\frac{\log P_{i,t-S}-\log P_{i,t-L}}{\sigma^{EWMAD}_{i,t}}
$$

Fundamental blend:

$$
Z^{fund}_{i,t}=\frac{\sum_k w_k Z^{(k)}_{i,t}}{\sum_k |w_k|}
$$

Combined alpha:

$$
\alpha_{i,t}=Z^{price}_{i,t}\bigl(1+\lambda Z^{fund}_{i,t}\bigr)
$$

The fundamental inputs include positive business momentum variables and negative deterioration variables such as leverage or accrual deterioration.

### Intuition

Pure price momentum is useful but incomplete. A stock with strong price momentum backed by improving revenue, margin, and operating cash flow is more credible than a stock with the same price momentum but worsening balance-sheet quality.

Why it can work:

- price trends often persist more when fundamentals confirm them
- the market underreacts to slowly released accounting information
- combining price and fundamentals reduces false-positive momentum trades

This is a good place to show researcher maturity: **I am not treating price and fundamentals as separate silos; I use fundamentals to qualify the persistence of price moves.**

## Robust Statistics: Why Median and MAD?

The repo uses robust cross-sectional standardization:

$$
z_i=\frac{x_i-\operatorname{median}(x)}{1.4826\cdot \operatorname{MAD}(x)}
$$

where:

$$
\operatorname{MAD}(x)=\operatorname{median}(|x_i-\operatorname{median}(x)|)
$$

Why this is better than mean/std in many equity panels:

- equity cross-sections are heavy-tailed
- accounting fields and returns both contain outliers
- naive mean/std scaling can be dominated by a few names

Strong interview line:

> Robust estimators are not a cosmetic choice. They change the effective geometry of the cross-section and make the portfolio less hostage to outliers.

## Regime Model: 2-State Gaussian HMM

### Step 1: Build a market proxy

The repo avoids fitting a latent-state model to every stock directly. Instead it constructs a robust market series from the cross-sectional median of winsorized log returns:

$$
m_t=\operatorname{median}_i(\text{winsorized log return}_{i,t})
$$

This compresses the panel into a low-noise market state signal.

### Step 2: Hidden-state model

Assume a latent state $s_t \in \{0,1\}$ with Markov transition matrix:

$$
A_{ij}=\Pr(s_t=j\mid s_{t-1}=i)
$$

Observation model:

$$
m_t \mid s_t=i \sim \mathcal{N}(\mu_i, \sigma_i^2)
$$

Initial state probabilities:

$$
\pi_i=\Pr(s_1=i)
$$

The model is fit by EM using forward-backward recursions.

### Step 3: Forward-backward math

Forward probability:

$$
\alpha_t(j)=\Pr(m_{1:t}, s_t=j)
$$

Backward probability:

$$
\beta_t(j)=\Pr(m_{t+1:T}\mid s_t=j)
$$

Posterior state probability:

$$
\gamma_t(j)=\Pr(s_t=j\mid m_{1:T}) \propto \alpha_t(j)\beta_t(j)
$$

Filtered probability used in the repo:

$$
p_t=\Pr(s_t=\text{up}\mid m_{1:t})
$$

Interview point: this is not a black box. It is a recursive Bayesian state-estimation problem.

### Step 4: EM intuition

In the E-step, estimate state probabilities using the current parameters.

In the M-step, update:

$$
\mu_i=\frac{\sum_t \gamma_t(i)m_t}{\sum_t \gamma_t(i)}
$$

$$
\sigma_i^2=\frac{\sum_t \gamma_t(i)(m_t-\mu_i)^2}{\sum_t \gamma_t(i)}
$$

and update the transition matrix using expected state transitions.

Why use HMM here:

- market states are persistent, not IID
- a hidden-state view captures clustering of calm vs stressed environments
- it gives a continuous probability, not a brittle hard label


### Step 5: Use HMM as an overlay, not a standalone alpha

The repo maps the filtered probability into a regime score:

$$
r_t=2p_t-1
$$

Then applies it to a base alpha:

$$
\alpha^{regime}_{i,t}=r_t\alpha^{base}_{i,t}
$$

Why this is a strong design:

- the stock-selection alpha remains the main engine
- the HMM controls **when** to trust or shrink that engine
- this avoids asking too much of the regime model

## Kalman Filter Overlay

The repo adds a Kalman filter after the HMM to stabilize regime scaling with a causal filter.

State equation:

$$
x_t=x_{t-1}+w_t, \quad w_t\sim\mathcal{N}(0,q)
$$

Observation equation:

$$
y_t=x_t+v_t, \quad v_t\sim\mathcal{N}(0,r)
$$

where $y_t$ is the raw regime scale from the HMM and $x_t$ is the filtered latent scale.


### Kalman Recursion

Predict:

$$
\hat{x}_{t|t-1}=\hat{x}_{t-1|t-1}
$$

$$
P_{t|t-1}=P_{t-1|t-1}+q
$$

Update:

$$
K_t=\frac{P_{t|t-1}}{P_{t|t-1}+r}
$$

$$
\hat{x}_{t|t}=\hat{x}_{t|t-1}+K_t(y_t-\hat{x}_{t|t-1})
$$

$$
P_{t|t}=(1-K_t)P_{t|t-1}
$$

Intuition:

- if the observation is noisy, do not fully trust it
- if uncertainty is high, adapt faster
- if the regime is stable, allow more persistence in the filtered scale


### Innovation Gating

The repo also uses the innovation z-score:

$$
z_t=\frac{y_t-\hat{x}_{t|t-1}}{\sqrt{P_{t|t-1}+r}}
$$

If $|z_t|$ is too large, the model interprets the observation as unusually surprising and reduces exposure toward a floor.

This is a strong practical detail. It says:

> When the market looks like it is changing state faster than my filter can interpret reliably, I reduce size rather than pretend I know the new regime with confidence.


## Beta Neutralization

The repo estimates rolling betas of each stock to a robust market series, then removes the projection of the weight vector onto the beta vector:

$$
w'_t=w_t-\frac{w_t^\top \beta_t}{\beta_t^\top \beta_t}\beta_t
$$

Why this matters:

- a stock-selection strategy can accidentally become a market-timing strategy
- if the signal's PnL is mostly market beta, the alpha is weak
- beta-neutralization makes attribution cleaner

Interview line:

> I want the book to earn from relative stock selection, not because it quietly drifted into a long market bet.

## Market Volatility Overlay

The repo computes a robust market-volatility z-score and shrinks exposure when volatility spikes:

$$
s_t=\operatorname{clip}(1-\gamma z_t^+, s_{\min},1)
$$

where $z_t^+=\max(z_t,0)$.

Why it works conceptually:

- very high market volatility often compresses cross-sectional signal quality
- transaction costs and slippage rise in stressed conditions
- a portfolio that keeps full size in every environment is usually overconfident

## Drawdown Overlay

Drawdown is computed from the equity curve:

$$
DD_t=\frac{EQ_t}{\max_{u\le t}EQ_u}-1
$$

Exposure is reduced once drawdown breaches a threshold.

This is less about prediction and more about portfolio discipline. It acts as a control against model drift, broken assumptions, or temporary regime mismatch.

## Why the Kinked Quality Function Matters

The repo does not optimize only Sharpe. Instead it uses a piecewise score that rewards metrics inside an acceptable range and penalizes metrics sharply outside that range.

For a metric $x$ with acceptable interval $[\ell,h]$:

- below $\ell$: steep penalty
- inside $[\ell,h]$: positive reward slope
- above $h$: smaller incremental reward

This reflects real research asymmetry:

- too much turnover is bad quickly
- too much concentration is bad quickly
- a slightly higher Sharpe above target is useful, but not infinitely useful

Interview line:

> I want my objective function to match how a PM actually thinks, not how a toy optimization problem thinks.

## Why This Repo Makes You Look Like a Good Researcher

- You use robust estimators instead of naive ones.
- You separate alpha, weighting, and overlays modularly.
- You measure implementation metrics explicitly.
- You think in cross-sectional portfolio terms, not single-name anecdotes.
- You use regime models as exposure controls, which is more defensible than pretending they forecast every detail.

## Good Interview Lines

- I prefer robust estimators because equity cross-sections are heavy-tailed and naive moments can be dominated by a handful of names.
- I optimize for deployable alpha quality, not only raw backtest return.
- I use regime models mainly to scale exposure, because that is where they are most reliable.
- I explicitly penalize turnover and max position size because attractive backtests often fail at those exact points.

## Weaknesses You Should Admit Honestly

- Grid search can overfit if validation design is weak.
- HMM states are interpretable but not guaranteed to be stable forever.
- Fundamental point-in-time alignment must be treated very carefully in production.
- Cost, borrow, and liquidity assumptions can always be improved.

Admitting these does not weaken your interview. It strengthens it, because it shows you understand where research turns into real portfolio risk.

## Likely Questions and Strong Answers

### Why HMM on the median market series?

Because it creates a low-noise, low-dimensional market state proxy. That is easier to fit robustly than a full high-dimensional hidden-state model on the stock panel.

### Why tail-gate the cross-section?

Because weak scores are mostly noise and create turnover without much information. Trading only the tails improves signal-to-noise and keeps the book focused on conviction.

### Why volatility scaling?

Because otherwise the portfolio is implicitly saying the noisiest names deserve the biggest risk budget. That is rarely what we want.

### Why use Kalman after HMM?

Because HMM probabilities can still be jittery. Kalman filtering produces a more stable exposure path and gives a principled way to identify when observations are unusually surprising.

### Why not just optimize Sharpe?

Because Sharpe alone ignores the shape of implementation risk. A strategy with good Sharpe but extreme turnover or concentration may be unusable.


## Final Framing

If you need one sentence to summarize the repo in an interview, use this:

> This repo shows that I think about alpha as a portfolio research problem, not just as a predictive-model problem.
